DROPOUT E BATCH NORMALIZATION: STABILIZZARE E REGOLARIZZARE LE CNN

Come rendere le nostre reti neurali robuste e voleci.
Quanto sono stabili le reti e quanto sono in grado di evitare le distrazioni.

- Batch Normalization: tecnica che permeette di addestrare reti profonde senza che i gradienti evaporano
- Dropout: le immagini hanno bisogno di una versione più intelligente
- Come incastrare questi layer in una pipeline keras

Come domare il caos dei dati
Man mano che una rete neurale diventa profonda, la distribuzione degli input ai lauyer interni cambia continuamente durante l'addestramento. Questo fenomeno rende difficile per i pesi adattarsi a un segnale che si sposta costantemente.
La Batch Normalization interviene forzando ogni mini-batch ad avere una distribuzione standard, normalizzando le attivazioni prima che queste passino alla funzione di attivazione successiva.

La BN risolve uno dei problemi più antichi del Deep Learning

Stabilizzazione dei Gradienti
Perche la normalizzazione previne il vanisching gradient, perche mantinene le attivazioni in una zona dove la relu non dorme mai.
- Riduzione della dipendenza dai pesi iniziali: la rete diventa meno sensibile alla strategia di inizializzazione grazie al ri-centramento dei dati
- Accelerazione della convergenza: consentendo l'uso di learning rate più elevati, la Batch Normalization ridure drasticamente il numero di epoche nesessarie.
- Effetto regolarizzante: l'aggiutna di una piccola componente stocastica dovuta alle statistiche del mini-batch aiuta a prevenire l'overfitting
- L'operazione trasforma l'input sottraendo la media dei batch e dividendo per la deviazione standard piùun termine di stabilità

Non dobbiamo più di avere pesi iniziali errati e possiamo utilizzare learning rate più agressivi arrivando a convergenza con meno epoche.

Parametri Appresi in BN
- Gamma e Beta: La BN impare gestendo due parametri Gamma e Beta, questo permette alla rete di decidere se quella noramlizzazione serve d'avvero o se preferisce tornare ai dati originali, annullando la normalizzazione (se necessario per l'apprendimento)
- Train vs Inference: durante l'inferenza, il layer non usa le statistiche del batch attuale, ma una media mobile calcolata durante l'intero processo di training.
- Stabilità numerica: L'introduzione di epsilon evita divisioni per zero in casi di batch con varianza nulla, garantendo che il calcolo del gradiente non fallisca mai.

L'Impatto sul Paesaggio della Loss
Levigare la superficie di ottimizzazione
Ricerche ricenti suggeriscono che il vero potere della BN non risieda solo nell controllo della distribuzione, ma nella capacità di rendere il paesaggio della funzione di costo più liscio.
Senza BN la strada verso il minimo globale è piena di buca e valli scoscese, la BN trasforma questo percorso tortuoso in una pianura navigabile, permetetndo ai gradienti di fluire in modo più veloce.
Una superficie di errore meno scoscesa permette all'ottimizzatore di navigare con passi più ampi e sicuri, evitando di restare intrappolati in minimi locali rumorosi.

Ora che abbiamo stabilizzato il flusso della BN, vediamo come impedire alla rete di imparare a memoria.

Evoluzione del Dropout
Dalla regolarizzazione puntuale a quella strutturale.
Il Dropout tradizionale spegne casualmente singolo neuroni per evitare che imparino a collaborare eccessivamente (co-adattamento). Tuttavia, nella immagini (nella CNN), i pixel vicino sono fortemente correlati (i pixel vicini si assomigliano).
Se spegnamo un solo pixel ma lasciamo accesi i suoi vicini, la rete può indovinare il valore mancante senza fare alcuno sforzo. Dobbiamo quindi passare ad un approccio più radicale, lo Special Dropout
Nelle reti convoluzionali è necessario un approccio diverso, passando dal Dropout standard alla versione Spaziale progettata per i volumi 3D.

Standard vs Spatial Dropout
Gestire la ridondanza spaziale della CNN
- Dropout Standard: agesce su singoli elementi indipendenti (agisce sul singolo pixel). Nelle feature map delle CNN, questo risulta spesso inefficiente perchè i neuroni adianceti portano la stessa informazione
- Spatial Dropout: spegnere interi canali (feature map) invece di singoli pixel, spegnendo un itero filtro (esempio quello che cerca i bordi veritcali). Se un canale viene spento, la rete è costratta a cercare l'informazione in altri filtri.
- Efficacia: la versione spaziale promuove l'indipendenza tra i filtri estratti, migliorando significativamente la robustezza del modello visivo. Garantisce che la conoscenza sia distribuita in modo uniforme.
- L'operazione di droout può essere vista come l'applicazione di una maschera binaria di Bernoulli al vettore delle attivazioni.

Regolarizzazione e Noise.
Sampling di Architetture: Il Dropout può essere interpretato come un modo per addestrare contemporaneamente un ensemble di milioni di reti neurali più piccole che condividono i pesi
Inference Modo: In fase di test, il Dropout viene disattivato e i pesi vengono scalalti per compensare la maggior potenza di segnale derivante dall'attivazione di tutti i neuroni.
Drop Rate: la scelta della probabilità (solitamente tra 0.2 e 0.5) determina il livello di 'stress' imposto alla rete durante l'apprendimento delle caratteristiche.

CNN e Spatial Invariance
Perchè spegnere un intero canale.
Poichè i layer convoluzionali estraggono feature spazialmente correlate, se spegniamo un solo pixel ma lasciamo attivi i suoi vicini, la rete può facilmente 'indovinare' il valore mancante.
Se un filtro impara a riconoscere una texture, quella texture sarà presente in centinaia di pixel vicini, con il Spatial Dropout eliminiamo la tendenza delle rete di trovare scorciatoie statistiche pigre.
Questo forzato disorientamente trasforma la rete di un detective più attento, obbligando a guardare alla rete da molteplici punti di vista.
Spegnendo l'intero canale con lo Spatial Dropout, forziamo il modello a non dipendere da un particolare rilevatore di bordi o texture, distribuendo la conoscenza in modo uniforme.

Come si cotruisce una rete robusta?

Implementazione Pratica
Configurazione di una rete robusta in Keras
La BN e il dropout possono convivere in uno stesso blocco per massimizzare sia la stabilità che la robustezza, preparando la rete a sfidare dataset complessi
Dove mettiamo la normalizzazione

Ordine dei Layer
Dove inserire la normalizzazione
- Post-Convolution: inserire BatchNormalization subito dopo il layer 'Conv2D' e prima della funzione di attivazione 'ReLu': Convoluzione + Batch Normalizatione e ReLu
- Post-Activation: alcuni ricercatori suggeriscono di inserire la BN dopo la 'ReLu'. Entrambi gli approcci sono validi, ma il primo è lo standard storico.
- Integrazione del Dropout: il dropout deve essere come un sigillo finale, solitamente inserito dopo il pooling e dopo i layer densi finali per regolarizzare l'output delle feature estratte, oppure nei layer densi finali per l'ultio tocco prima del verdetto della softmax.
- Il numero di parametri di un layer di Batch Normalization è pari a quattro volte il numero di filtri del layer precedente.

Sintassi in Keras
Una sola riga di codice layer.batchnormalization.
Per le immagini ricordardi di usare layer.spatialedropout2d (non usare lo standard)

Ottimizzazoine del Workflow
Consigli per il design
Quando usate la Batch Normalization, subito dopo la convoluzione, potete permettervi di rimuovere il termine 'bias' dal layer convoluzione precedente ('use_bias=False'), poichè la BN include già un parametri di shift.
Questo piccolo accorgimento, riduce leggermente la ridondanza dei parametri senza influire sulle prestazioni finali della rete.






In [1]:
import keras
from keras import layers, models

def build_robust_cnn_2025(input_shape=(32, 32, 3)):
    """
    Costruisce una CNN ottimizzata con tecniche di regolarizzazione avanzata.
    Ideale per dataset come CIFAR-10 o MNIST (previo resize).
    """
    model = models.Sequential(name="Robust_CNN_2025")

    # 2. DEFINIZIONE ESPLICITA DELL'INPUT
    # A differenza del passato, definire l'Input layer è best practice per:
    # - Validazione immediata dei tensori (Eager Execution).
    # - Debugging della forma (shape) prima della compilazione.
    model.add(layers.Input(shape=input_shape))

    # --- BLOCCO 1: ESTRAZIONE FEATURE E STABILIZZAZIONE ---
    # use_bias=False: Quando segue una BatchNormalization (BN), il parametro 'bias' 
    # della convoluzione è matematicamente ridondante. La BN applica un termine 
    # di 'beta' (offset) che svolge la stessa funzione. Rimuoverlo riduce i parametri totali.
    model.add(layers.Conv2D(32, (3, 3), padding='same', use_bias=False))
    
    # BatchNormalization: Fondamentale per combattere l'Internal Covariate Shift.
    # Normalizza l'output del layer precedente, accelerando la convergenza 
    # e agendo come una leggera forma di regolarizzazione.
    model.add(layers.BatchNormalization()) 
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    
    # --- BLOCCO 2: REGOLARIZZAZIONE SPAZIALE ---
    model.add(layers.Conv2D(64, (3, 3), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    
    # SpatialDropout2D vs Dropout standard:
    # Nelle CNN, i pixel adiacenti sono fortemente correlati. Il Dropout standard 
    # spegne pixel casuali, ma l'informazione "sopravvive" tramite i vicini. 
    # SpatialDropout spegne interi canali (feature maps), costringendo la rete 
    # a non fare affidamento su specifici filtri per riconoscere un pattern.
    model.add(layers.SpatialDropout2D(0.3)) 

    # --- TESTA DEL MODELLO (CLASSIFICATORE) ---
    # Flatten trasforma il tensore 3D in un vettore 1D per i layer densi.
    model.add(layers.Flatten())
    
    # Dense Layer con Dropout: Qui usiamo il Dropout standard (0.5).
    # È la difesa finale contro l'overfitting nei layer fully-connected, 
    # dove risiede la maggior parte dei parametri addestrabili.
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5)) 
    
    # Output Layer: Softmax garantisce che la somma delle probabilità delle 10 classi sia 1.0.
    model.add(layers.Dense(10, activation='softmax'))

    return model

# Istanziamo il modello
model = build_robust_cnn_2025()

# 3. ISPEZIONE DEL SUMMARY
# Nota: Nel summary vedrai "Non-trainable params" nei layer di BatchNormalization.
# Questi rappresentano la 'media' e la 'varianza' mobile calcolate durante il training,
# che verranno "congelate" e usate durante l'inferenza (predizione).
model.summary()

Model: "Robust_CNN_2025"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d               │ (None, 8, 8, 64)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,386 (2.08 MB)

 Trainable params: 545,194 (2.08 MB)

 Non-trainable params: 192 (768.00 B)

la BN stabilizza i dati e accellare il learning rete.
Il dropuut spatial spegne interi canali